# EZStats - Train Pitch Keypoint Detector v2  (#3 - MED ROI)

**What this fixes (partly):** the 2D-map 'jump' comes from per-frame keypoint jitter. Higher input resolution = more precise keypoints = less jitter. The *remaining* jitter is fixed in code (homography averaging in `stats_video.py`, deferred).

**Same data, better training:**

| | Old (V2 model) | This v2-hires |
|---|---|---|
| Resolution | **640** | **1280** (main lever for keypoint precision) |
| Epochs | 100 | **200 + early stop** |
| LR | linear | **cosine** |
| Mosaic | 0.0 (off) | 0.0 (off - mosaic corrupts keypoint geometry, keep it off!) |

> Note: keypoint augmentation is delicate. We keep `mosaic=0.0` and `fliplr=0.0` because horizontal flip needs a symmetric keypoint index map; getting it wrong silently ruins the labels. Resolution + epochs are the safe, effective levers here.

---
### Run order: A -> B -> C -> D -> E -> F -> G.

## A - Turn on the GPU
**Runtime -> Change runtime type -> GPU -> Save.** imgsz 1280 pose training is heavy - A100 preferred, T4 works but slow. Run the cell.

In [ ]:
!nvidia-smi

## B - Connect Google Drive (resume-safe)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
HOME = os.getcwd()
RUNS_DIR = Path('/content/drive/MyDrive/ezstats/runs')
RUN_NAME = 'pitch_keypoints_v2_hires'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Saving to:', RUNS_DIR / RUN_NAME)

## C - Install

In [ ]:
!pip install -q ultralytics roboflow

## D - Roboflow key + download the SAME pitch dataset
Add secret `ROBOFLOW_API_KEY` (key icon, left sidebar), then run.

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace('roboflow-jvuqo').project('football-field-detection-f07vi')
version = project.version(12)
dataset = version.download('yolov8')
print('Downloaded to:', dataset.location)

In [ ]:
!sed -i 's|\(train: \).*|\1../train/images|' {dataset.location}/data.yaml
!sed -i 's|\(val: \).*|\1../valid/images|' {dataset.location}/data.yaml
!cat {dataset.location}/data.yaml

## E - Train (pose, resume-safe)
imgsz=1280 + 200 epochs with early stop. Mosaic OFF (required for keypoints). Re-run after any disconnect to resume.

In [ ]:
%cd {HOME}

ckpt = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
if ckpt.exists():
    print('Found checkpoint -> RESUMING from', ckpt)
    !yolo task=pose mode=train resume=True model='{ckpt}'
else:
    print('Fresh training run.')
    !yolo task=pose mode=train \
      model=yolov8x-pose.pt \
      data={dataset.location}/data.yaml \
      epochs=200 \
      imgsz=1280 \
      batch=6 \
      patience=50 \
      cos_lr=True \
      mosaic=0.0 fliplr=0.0 \
      translate=0.1 scale=0.4 \
      plots=True \
      project='{RUNS_DIR}' name='{RUN_NAME}'

## F - Check results
Watch **Pose mAP50-95** (the old model was ~0.47 on only 30 val images). Higher = more precise keypoints. The real proof is on YOUR video though - validate before swapping.

In [ ]:
%cd {HOME}
!yolo task=pose mode=val \
  model='{RUNS_DIR}/{RUN_NAME}/weights/best.pt' \
  data={dataset.location}/data.yaml imgsz=1280

In [ ]:
from IPython.display import Image
Image(filename=f'{RUNS_DIR}/{RUN_NAME}/results.png', width=900)

In [ ]:
Image(filename=f'{RUNS_DIR}/{RUN_NAME}/val_batch0_pred.jpg', width=900)

## G - Done - model on Drive
`MyDrive/ezstats/runs/pitch_keypoints_v2_hires/weights/best.pt`

On laptop: download -> `artifacts/pitch/football-pitch-detectionV2-hires.pt` (NEW name, keep old V2). Tell Claude to validate the 2D map on `08fd33_4.mp4` before swapping.